# Étape 3 : contrôles de format et de complétude

Ce notebook applique aux données chargées les règles du dictionnaire enregistré (`data/config/dictionnaire.yaml`), tel qu'il a été généré puis revu par le métier dans le notebook `02c_dictionnaire`.

Pour chaque colonne, c'est la règle en vigueur qui est contrôlée : champ obligatoire, nature, liste de valeurs, format. Un écart est une anomalie, ou reste à arbitrer quand le métier n'a pas encore décidé de la valeur ou de la forme. Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `02c_dictionnaire`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 02c_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
%%time
# CONTROLES

from frigeo.controles import controler, regles_sans_controle

ecarts, bilan = controler(tables, dictionnaire)
sans_controle = regles_sans_controle(dictionnaire)
print(len(bilan), "contrôles,", len(sans_controle), "règles sans contrôle")
print(len(ecarts), "écarts, dont", int(bilan["anomalies"].sum()), "anomalies et",
      int(bilan["a_arbitrer"].sum()), "à arbitrer")
print(int(bilan["non_revues"].sum()), "cellules admises mais non revues")
display(
    bilan.groupby(["regle", "statut_regle"])[
        ["controlees", "anomalies", "a_arbitrer", "non_revues"]
    ].sum()
)

In [ ]:
# DETAIL PAR CONTROLE

a_voir = bilan[(bilan["anomalies"] > 0) | (bilan["a_arbitrer"] > 0)]
print(len(a_voir), "contrôles avec des anomalies ou des écarts à arbitrer")
display(a_voir.drop(columns=["non_revues"]))

# Une règle « à décider » ne contrôle rien tant que le métier n'a pas tranché.
a_decider = sans_controle[sans_controle["regle_en_vigueur"] == "à décider"]
print(len(a_decider), "règles « à décider », colonnes non contrôlées sur ce point")
display(a_decider)

In [ ]:
# COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite
from frigeo.controles import VALEUR_MASQUEE

sensibles = charger_sensibilite()
sur_sensible = pd.Series(
    [colonne in sensibles.get(table, ()) for table, colonne in zip(ecarts["table"], ecarts["colonne"])],
    index=ecarts.index,
    dtype=bool,
)
print(sum(len(colonnes) for colonnes in sensibles.values()), "colonnes sensibles,",
      int(sur_sensible.sum()), "écarts sur ces colonnes")
print("valeurs non masquées sur ces colonnes", int((ecarts.loc[sur_sensible, "valeur"] != VALEUR_MASQUEE).sum()))
print("valeurs masquées sur les autres colonnes", int((ecarts.loc[~sur_sensible, "valeur"] == VALEUR_MASQUEE).sum()))
display(
    ecarts[sur_sensible].groupby(["table", "colonne", "regle", "constat"]).size().rename("ecarts").to_frame()
)